# SparkSession Creation and Data Reading

In [13]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Instacart Sales Analysis") \
    .config("spark.driver.memory", "5g") \
    .config("spark.executor.memory", "5g") \
    .getOrCreate()

print("Spark started")

df = spark.read.parquet("instamart_dataset.parquet")

Spark started


# Shape of Dataset

In [14]:
df.count(), len(df.columns)

(32434489, 15)

In [15]:
df.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- add_to_cart_order: long (nullable = true)
 |-- reordered: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- eval_set: string (nullable = true)
 |-- order_number: long (nullable = true)
 |-- order_dow: long (nullable = true)
 |-- order_hour_of_day: long (nullable = true)
 |-- days_since_prior_order: double (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle_id: long (nullable = true)
 |-- department_id: long (nullable = true)
 |-- aisle: string (nullable = true)
 |-- department: string (nullable = true)



# Null Values Check

The null values are expected in the column 'days_since_prior_order' because there are customers who are buying first time from the store.

We need to replace nulls values with 0 or 'first_time_order'.

In [16]:
from pyspark.sql.functions import col, when, count

df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

+--------+----------+-----------------+---------+-------+--------+------------+---------+-----------------+----------------------+------------+--------+-------------+-----+----------+
|order_id|product_id|add_to_cart_order|reordered|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|product_name|aisle_id|department_id|aisle|department|
+--------+----------+-----------------+---------+-------+--------+------------+---------+-----------------+----------------------+------------+--------+-------------+-----+----------+
|       0|         0|                0|        0|      0|       0|           0|        0|                0|               2078068|           0|       0|            0|    0|         0|
+--------+----------+-----------------+---------+-------+--------+------------+---------+-----------------+----------------------+------------+--------+-------------+-----+----------+



# FEATURE ENGINEERING

In [18]:
from pyspark.sql.functions import col

df = df.withColumn("order_dow",col("order_dow").cast("int"))

df = df.withColumn("order_hour_of_day",col("order_hour_of_day").cast("int"))

df = df.fillna({"days_since_prior_order":0})

In [19]:
from pyspark.sql.functions import when, col

df = df.withColumn(
    "day_name",
    when(col("order_dow") == 0, "Sunday")
    .when(col("order_dow") == 1, "Monday")
    .when(col("order_dow") == 2, "Tuesday")
    .when(col("order_dow") == 3, "Wednesday")
    .when(col("order_dow") == 4, "Thursday")
    .when(col("order_dow") == 5, "Friday")
    .when(col("order_dow") == 6, "Saturday")
)

In [20]:
df.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- add_to_cart_order: long (nullable = true)
 |-- reordered: boolean (nullable = true)
 |-- user_id: long (nullable = true)
 |-- eval_set: string (nullable = true)
 |-- order_number: long (nullable = true)
 |-- order_dow: integer (nullable = true)
 |-- order_hour_of_day: integer (nullable = true)
 |-- days_since_prior_order: double (nullable = false)
 |-- product_name: string (nullable = true)
 |-- aisle_id: long (nullable = true)
 |-- department_id: long (nullable = true)
 |-- aisle: string (nullable = true)
 |-- department: string (nullable = true)
 |-- day_name: string (nullable = true)

